<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/AAI_MIXTRAL_TOPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install compressed-tensors>=0.15.0 -q

In [ ]:
!pip show compressed-tensors transformers torch

Name: compressed-tensors
Version: 0.17.1
Summary: Library for utilization of compressed safetensors of neural network models
Home-page: https://github.com/vllm-project/compressed-tensors
Author: The vLLM Project
Author-email: vllm-questions@lists.berkeley.edu
License: Apache 2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: loguru, pydantic, torch, transformers
Required-by: 
---
Name: transformers
Version: 5.10.2
Summary: Transformers: the model-definition framework for state-of-the-art machine learning models in text, vision, audio, and multimodal models, for both inference and training.
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Requires: huggingface-hub, numpy, packaging, pyyaml, regex, saf

In [ ]:
# ============================================================================
# REAL MULTI-AGENT SYSTEM WITH ACTUAL MIXTRAL MODEL LOADING - CORRECTED v2
# Head architecture, dtype, activation, bias-correction and fallback logic
# now match the reference notebook (MIXTRAL_TOPO_INFERENCE).
#
# Root cause of the earlier "heads NONE" failure:
#   The checkpoint heads are nn.Sequential (Linear->GELU->Dropout->Linear),
#   with keys classifier_X.0.* and classifier_X.3.*, but the heads were
#   defined as a single nn.Linear (classifier_X.weight). Keys didn't match,
#   so load_state_dict(strict=False) silently dropped all 12 tensors.
# ============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import hf_hub_download
import numpy as np
import json
import gc
import random
import time
from typing import Any
from dataclasses import dataclass
from enum import Enum

# ============================================================================
# SET SEED
# ============================================================================
def set_seed(seed=123):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(123)
print("🎲 Random seed set to 123")

# ============================================================================
# CONFIGURATION  (mirrors the reference notebook)
# ============================================================================
HF_REPO_ID    = 'frankmorales2020/topological-ai-mixtral-8x7b-multirun'
BASE_MODEL_ID = 'frankmorales2020/mixtral-8x7b-fp8-topo2026'
HIDDEN_SIZE   = 4096
HEAD_HIDDEN   = 512
DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'

TASK_CONFIG = {
    'A': {'name': 'World vs Sports',     'labels': {0: 'World', 1: 'Sports'},
          'confidence_threshold': 0.75, 'fallback_task': None, 'bias_correction': 0.0},
    'B': {'name': 'Business vs Sci/Tech','labels': {0: 'Business', 1: 'Sci/Tech'},
          'confidence_threshold': 0.75, 'fallback_task': None, 'bias_correction': 0.0},
    'C': {'name': 'World vs Sci/Tech',   'labels': {0: 'World', 1: 'Sci/Tech'},
          'confidence_threshold': 0.80, 'fallback_task': 'B', 'bias_correction': -0.5},
}

# ============================================================================
# DISABLE COMPRESSED TENSORS HOOKS
# ============================================================================
def disable_compressed_tensors_hooks(model):
    for module in model.modules():
        for hook_dict_name in ('_forward_pre_hooks', '_forward_hooks'):
            hook_dict = getattr(module, hook_dict_name, None)
            if hook_dict:
                to_remove = [hid for hid, hook in hook_dict.items()
                             if 'compress' in str(hook).lower() or 'decompress' in str(hook).lower()]
                for hid in to_remove:
                    del hook_dict[hid]
    return model

# ============================================================================
# LOAD REAL MODEL
# ============================================================================
print("=" * 70)
print("LOADING REAL MIXTRAL MODEL")
print("=" * 70)

print(f"📥 Loading base model from {BASE_MODEL_ID}...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    trust_remote_code=True,
    dtype=torch.bfloat16,
    device_map=None,
    low_cpu_mem_usage=False,
    ignore_mismatched_sizes=True,
)
base_model = base_model.cpu()

print("🔧 Removing compression hooks...")
base_model = disable_compressed_tensors_hooks(base_model)

print("🔄 Converting FP8 tensors to bfloat16...")
converted = 0
for _, param in base_model.named_parameters():
    if param.dtype in (torch.float8_e4m3fn, torch.float8_e5m2):
        param.data = param.data.to(torch.bfloat16); converted += 1
for _, buffer in base_model.named_buffers():
    if buffer.dtype in (torch.float8_e4m3fn, torch.float8_e5m2):
        buffer.data = buffer.data.to(torch.bfloat16); converted += 1
print(f"   Converted {converted} FP8 tensors")

print("🧹 Removing compression attributes...")
attrs_removed = 0
for module in base_model.modules():
    for attr in ['input_scale', 'weight_scale', 'down_proj_scale', 'gate_up_proj_scale',
                 'scales', 'zero_points', 'quantization_scheme', 'compression_config']:
        if hasattr(module, attr):
            delattr(module, attr); attrs_removed += 1
print(f"   Removed {attrs_removed} compression attributes")

gc.collect()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

print("📚 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ============================================================================
# TASK-AWARE MODEL  (head arch matches the checkpoint: Sequential .0/.1/.2/.3)
# ============================================================================
class RealTaskAwareModel(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model
        for param in self.base_model.parameters():
            param.requires_grad = False

        def _head():
            return nn.Sequential(
                nn.Linear(HIDDEN_SIZE, HEAD_HIDDEN, dtype=torch.bfloat16),  # .0
                nn.GELU(),                                                  # .1
                nn.Dropout(0.1),                                            # .2
                nn.Linear(HEAD_HIDDEN, 2, dtype=torch.bfloat16),           # .3
            )

        self.classifier_A = _head()
        self.classifier_B = _head()
        self.classifier_C = _head()
        self.current_task = 'A'
        self.loaded_heads = set()   # populated after verified checkpoint load
        self._init_heads()

    def _init_heads(self):
        for head in (self.classifier_A, self.classifier_B, self.classifier_C):
            for layer in head:
                if isinstance(layer, nn.Linear):
                    nn.init.xavier_uniform_(layer.weight)
                    if layer.bias is not None:
                        nn.init.zeros_(layer.bias)

    def switch_task(self, task):
        self.current_task = task

    def forward(self, input_ids, attention_mask=None):
        with torch.no_grad():
            outputs = self.base_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True,
                use_cache=False,
            )
            hidden = outputs.hidden_states[-1]

        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).to(hidden.dtype)
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        else:
            pooled = hidden.mean(dim=1)

        classifier = getattr(self, f'classifier_{self.current_task}')
        return classifier(pooled)

    @torch.no_grad()
    def predict_task(self, text, task='A'):
        """Single-head prediction with the task's bias correction applied."""
        inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=256)
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        self.switch_task(task)
        logits = self.forward(inputs['input_ids'], inputs['attention_mask'])

        bias = TASK_CONFIG[task].get('bias_correction', 0.0)
        if bias != 0.0:
            logits[:, 0] += bias

        probs = F.softmax(logits.float(), dim=-1)
        pred_class = probs.argmax().item()
        confidence = probs.max().item()
        label = TASK_CONFIG[task]['labels'][pred_class]
        return label, confidence, pred_class

    @torch.no_grad()
    def predict_with_fallback(self, text, task='C'):
        """Primary prediction; fall back to fallback_task if below threshold."""
        cfg = TASK_CONFIG[task]
        label, conf, cls = self.predict_task(text, task)
        if conf >= cfg['confidence_threshold'] or not cfg.get('fallback_task'):
            return label, conf, 'primary'
        fb = cfg['fallback_task']
        fb_label, fb_conf, _ = self.predict_task(text, fb)
        return fb_label, fb_conf, f'fallback->{fb}'

# ============================================================================
# CREATE MODEL + VERIFIED CHECKPOINT LOAD
# ============================================================================
print("\n" + "=" * 70)
print("CREATING TASK-AWARE MODEL")
print("=" * 70)

real_model = RealTaskAwareModel(base_model)

def head_signature(model, name):
    seq = getattr(model, name)
    sig = []
    for layer in seq:
        if isinstance(layer, nn.Linear):
            sig.append(round(layer.weight.float().abs().sum().item(), 4))
            sig.append(round(layer.bias.float().abs().sum().item(), 4))
    return tuple(sig)

pre_load_sig = {n: head_signature(real_model, n)
                for n in ('classifier_A', 'classifier_B', 'classifier_C')}

print("📦 Attempting to load certified weights...")
try:
    checkpoint_path = hf_hub_download(repo_id=HF_REPO_ID, filename='certified_topological_best.pt')
    checkpoint = torch.load(checkpoint_path, map_location='cpu')

    # Match checkpoint dtype to the bfloat16 heads.
    for k, v in checkpoint.items():
        if isinstance(v, torch.Tensor) and v.dtype == torch.float32:
            checkpoint[k] = v.to(torch.bfloat16)

    filtered = {k: v for k, v in checkpoint.items() if k.startswith('classifier_')}
    missing, unexpected = real_model.load_state_dict(filtered, strict=False)
    print(f"✓ Filtered {len(filtered)} classifier tensors from checkpoint")

    for name in ('classifier_A', 'classifier_B', 'classifier_C'):
        if head_signature(real_model, name) != pre_load_sig[name]:
            real_model.loaded_heads.add(name[-1])

    trusted = sorted(real_model.loaded_heads)
    untrusted = [t for t in ('A', 'B', 'C') if t not in real_model.loaded_heads]
    print(f"  ✅ Trained heads loaded: {trusted if trusted else 'NONE'}")
    if untrusted:
        print(f"  ⚠️  Heads still at RANDOM init: {untrusted}")
    cls_unexpected = [k for k in unexpected if 'classifier' in k]
    if cls_unexpected:
        print(f"  ⚠️  Unexpected classifier keys (arch mismatch): {cls_unexpected[:6]}")
except Exception as e:
    print(f"⚠️ Could not load certified weights: {e}")
    print("   Using randomly initialized classifiers — outputs are NOT meaningful.")

del pre_load_sig
print(f"📱 Moving model to {DEVICE}...")
real_model = real_model.to(DEVICE)
real_model.eval()
gc.collect()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

print(f"✓ Base dtype: {next(real_model.base_model.parameters()).dtype}")
print(f"✓ Head dtype: {next(real_model.classifier_A.parameters()).dtype}")

# ============================================================================
# AGENTS
# ============================================================================
class AgentRole(Enum):
    CLASSIFIER = "classifier"
    TOPIC = "topic"
    SENTIMENT = "sentiment"
    DECISION = "decision"

@dataclass
class AgentOutput:
    agent: str
    result: Any
    confidence: float

class RealClassifierAgent:
    """
    The three heads are binary classifiers on different axes, so voting across
    them is invalid (Task B can't express "Sports", etc.). Instead we ROUTE each
    document to the single head whose axis fits, using keyword signals, then
    trust only that head. Task C keeps its bias-correction + fallback-to-B.
    """
    # Keyword signals -> which binary head can actually decide this document.
    ROUTING = {
        'A': ['game', 'team', 'win', 'won', 'championship', 'league', 'match',
              'victory', 'series', 'cup', 'tournament', 'season', 'player',
              'coach', 'score', 'playoff'],
        'B': ['stock', 'earnings', 'revenue', 'sales', 'market', 'funding',
              'profit', 'shares', 'quarterly', 'investor', 'ipo', 'merger'],
    }

    def __init__(self, model):
        self.model = model

    # Health/science/general-news terms belong on the World axis; if these are
    # present we route to C and suppress the B fallback (B can't express World).
    WORLD_SIGNAL = ['cancer', 'treatment', 'medical', 'drug', 'disease', 'patient',
                    'vaccine', 'health', 'president', 'government', 'election',
                    'war', 'climate', 'court', 'nasa', 'space', 'mission']

    def _route(self, text):
        tl = text.lower()
        a_hits = sum(1 for kw in self.ROUTING['A'] if kw in tl)
        b_hits = sum(1 for kw in self.ROUTING['B'] if kw in tl)
        w_hits = sum(1 for kw in self.WORLD_SIGNAL if kw in tl)
        if a_hits and a_hits >= b_hits:
            return 'A'
        # Strong world/health/science signal outweighs a stray business keyword.
        if w_hits and w_hits >= b_hits:
            return 'C'
        if b_hits:
            return 'B'
        return 'C'

    def process(self, text):
        task = self._route(text)

        if task == 'C':
            tl = text.lower()
            has_world = any(kw in tl for kw in self.WORLD_SIGNAL)
            if has_world:
                # Trust C directly; B's fallback can't express World and would
                # mislabel general/health news as Business/Sci-Tech.
                label, conf, _ = self.model.predict_task(text, 'C')
                source = 'primary'
            else:
                label, conf, source = self.model.predict_with_fallback(text, 'C')
        else:
            label, conf, _ = self.model.predict_task(text, task)
            source = 'primary'

        return AgentOutput(
            agent='Classifier',
            result={
                'category': label,
                'routed_task': task,
                'task_c_source': source,
            },
            confidence=conf
        )

class RealTopicAgent:
    def process(self, text):
        topics = []
        text_lower = text.lower()
        topic_keywords = {
            'sports': ['game', 'team', 'win', 'championship', 'league', 'match', 'victory'],
            'business': ['stock', 'earnings', 'revenue', 'sales', 'market', 'funding', 'profit'],
            'technology': ['ai', 'quantum', 'software', 'cloud', 'algorithm', 'chip', 'computing'],
            'health': ['cancer', 'treatment', 'medical', 'drug', 'disease', 'breakthrough'],
            'politics': ['president', 'government', 'policy', 'election', 'law', 'congress'],
            'space': ['nasa', 'mission', 'space', 'moon', 'mars', 'jupiter', 'launch'],
        }
        for topic, keywords in topic_keywords.items():
            if any(kw in text_lower for kw in keywords):
                topics.append(topic)
        return AgentOutput(agent='Topic', result=topics if topics else ['general'],
                           confidence=0.8 if topics else 0.5)

class RealSentimentAgent:
    def process(self, text):
        text_lower = text.lower()
        positive = ['won', 'victory', 'breakthrough', 'growth', 'strong', 'record', 'beat', 'secured', 'surged']
        negative = ['loss', 'defeat', 'crisis', 'decline', 'risk', 'concern', 'failed']
        pos = sum(1 for w in positive if w in text_lower)
        neg = sum(1 for w in negative if w in text_lower)
        if pos > neg:
            return AgentOutput(agent='Sentiment', result='positive', confidence=0.8)
        elif neg > pos:
            return AgentOutput(agent='Sentiment', result='negative', confidence=0.8)
        return AgentOutput(agent='Sentiment', result='neutral', confidence=0.5)

class RealDecisionAgent:
    def process(self, classifier_out, topic_out, sentiment_out):
        category = classifier_out.result['category']
        confidence = classifier_out.confidence
        source = classifier_out.result.get('task_c_source', 'primary')
        is_fallback = source.startswith('fallback')
        # Auto-approve only confident, non-fallback predictions.
        if confidence >= 0.75 and not is_fallback:
            action, review = 'auto_approve', False
        else:
            action, review = 'review', True
        routing = {
            'Sports': 'sports_desk', 'Business': 'finance_team',
            'Sci/Tech': 'tech_research', 'World': 'news_department',
        }.get(category, 'general_inbox')
        return AgentOutput(
            agent='Decision',
            result={'category': category, 'action': action, 'routing': routing,
                    'requires_review': review, 'topics': topic_out.result,
                    'sentiment': sentiment_out.result},
            confidence=confidence
        )

# ============================================================================
# ORCHESTRATOR
# ============================================================================
class RealOrchestrator:
    def __init__(self, model):
        self.classifier = RealClassifierAgent(model)
        self.topic = RealTopicAgent()
        self.sentiment = RealSentimentAgent()
        self.decision = RealDecisionAgent()

    def analyze(self, text, verbose=True):
        start = time.time()
        classifier_out = self.classifier.process(text)
        topic_out = self.topic.process(text)
        sentiment_out = self.sentiment.process(text)
        decision_out = self.decision.process(classifier_out, topic_out, sentiment_out)
        elapsed = (time.time() - start) * 1000

        if verbose:
            c = classifier_out.result
            print(f"\n{'=' * 55}")
            print(f"📝 {text[:65]}...")
            print(f"{'=' * 55}")
            print(f"🎯 {decision_out.result['category']} ({classifier_out.confidence * 100:.0f}%)"
                  f"  [routed->Task {c['routed_task']}, source {c['task_c_source']}]")
            print(f"🏷️  {', '.join(topic_out.result[:3])}")
            print(f"💭 {sentiment_out.result}")
            print(f"✅ {decision_out.result['action']} → {decision_out.result['routing']}")
            print(f"⏱️  {elapsed:.0f}ms")

        return {
            'text': text[:100],
            'category': decision_out.result['category'],
            'confidence': classifier_out.confidence,
            'routed_task': classifier_out.result['routed_task'],
            'task_c_source': classifier_out.result['task_c_source'],
            'topics': topic_out.result,
            'sentiment': sentiment_out.result,
            'action': decision_out.result['action'],
            'routing': decision_out.result['routing'],
            'time_ms': elapsed,
        }

# ============================================================================
# RUN
# ============================================================================
print("\n" + "=" * 70)
print("RUNNING REAL MULTI-AGENT SYSTEM")
print("=" * 70)

orchestrator = RealOrchestrator(real_model)

test_docs = [
    "The Chicago Cubs won the World Series championship",
    "Microsoft stock surged 15% after beating earnings",
    "NASA launched a mission to explore Jupiter's moon Europa",
    "Scientists discovered breakthrough treatment for cancer",
    "Apple reported record iPhone sales and new AI features",
]

results = []
for doc in test_docs:
    results.append(orchestrator.analyze(doc, verbose=True))

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
for i, r in enumerate(results, 1):
    icon = "✅" if r['action'] == 'auto_approve' else "⚠️"
    print(f"{icon} {i}. {r['category']:10s} | {r['action']:12s} | "
          f"{r['confidence'] * 100:.0f}% (Task {r['routed_task']})")

with open('real_model_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("\n💾 Saved to 'real_model_results.json'")
print("\n✅ REAL MODEL SYSTEM READY")

🎲 Random seed set to 123
LOADING REAL MIXTRAL MODEL
📥 Loading base model from frankmorales2020/mixtral-8x7b-fp8-topo2026...


Compressing model: 100%|██████████| 128/128 [00:00<00:00, 1972.75it/s]


Loading weights:   0%|          | 0/547 [00:00<?, ?it/s]

[transformers] MixtralForCausalLM LOAD REPORT from: frankmorales2020/mixtral-8x7b-fp8-topo2026
Key                                                                       | Status     |  | 
--------------------------------------------------------------------------+------------+--+-
model.layers.{0...31}.mlp.experts.{0, 1, 2, 3, 4, 5, 6, 7}.w3.input_scale | UNEXPECTED |  | 
model.layers.{0...31}.mlp.experts.{0, 1, 2, 3, 4, 5, 6, 7}.w1.input_scale | UNEXPECTED |  | 
model.layers.{0...31}.mlp.experts.{0, 1, 2, 3, 4, 5, 6, 7}.w2.input_scale | UNEXPECTED |  | 
model.layers.{0...31}.mlp.experts.gate_up_proj_scale                      | UNEXPECTED |  | 
model.layers.{0...31}.mlp.experts.down_proj_scale                         | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔧 Removing compression hooks...
🔄 Converting FP8 tensors to bfloat16...
   Converted 192 FP8 tensors
🧹 Removing compression attributes...
   Removed 384 compression attributes
📚 Loading tokenizer...

CREATING TASK-AWARE MODEL
📦 Attempting to load certified weights...
✓ Filtered 12 classifier tensors from checkpoint
  ✅ Trained heads loaded: ['A', 'B', 'C']
📱 Moving model to cuda...
✓ Base dtype: torch.bfloat16
✓ Head dtype: torch.bfloat16

RUNNING REAL MULTI-AGENT SYSTEM

📝 The Chicago Cubs won the World Series championship...
🎯 Sports (92%)  [routed->Task A, source primary]
🏷️  sports
💭 positive
✅ auto_approve → sports_desk
⏱️  1154ms

📝 Microsoft stock surged 15% after beating earnings...
🎯 Business (98%)  [routed->Task B, source primary]
🏷️  business
💭 positive
✅ auto_approve → finance_team
⏱️  67ms

📝 NASA launched a mission to explore Jupiter's moon Europa...
🎯 World (94%)  [routed->Task C, source primary]
🏷️  space
💭 neutral
✅ auto_approve → news_department
⏱️  67ms

📝 Scientists